In [ ]:
!pip install GenCareAI
from GenCareAI.GenCareAIUtils import GenCareAISetup

setup = GenCareAISetup()

if setup.environment == 'Colab':
        !pip install -q langchain langchain-openai langchain-community chromadb datasets langchain-chroma

In [2]:
# Import necessary modules from Langchain and Hugging Face
import os
import pandas as pd
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings, OpenAI
from langchain_community.document_loaders import HuggingFaceDatasetLoader, DataFrameLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.schema.document import Document
from langchain.chains import RetrievalQA
from pprint import pprint

In [3]:
# Constants for dataset and storage paths
PATH_DATASET = setup.get_file_path('data/chat_with_client_records/df_cr.csv') 
PATH_DB_GCAI = setup.get_file_path('data/chroma_db_gcai_notes')
COLLECTION_NAME = 'chat_with_records'
MODEL = 'text-embedding-ada-002'

In [ ]:
def load_documents(path, content_column_name):
    """Load the dataset either from Hugging Face or a local CSV file based on the path provided."""
    
    try:
        # Try to load the Hugging Face dataset       
        loader = HuggingFaceDatasetLoader(path, page_content_column=content_column_name, use_auth_token=setup.get_hf_token())
        return loader.load()
    
    except Exception:
        # If loading as a Hugging Face dataset fails, assume it's a CSV file        
        df = pd.read_csv(path, index_col=0)
        loader = DataFrameLoader(df, page_content_column=content_column_name, )
        return loader.load()

documents = load_documents(path=PATH_DATASET, content_column_name='note')

In [ ]:
documents

In [ ]:
def split_documents(documents: list[Document]):
  """Split large text documents into manageable chunks for better handling by ML models."""
  text_splitter = RecursiveCharacterTextSplitter(
      chunk_size=800,
      chunk_overlap=100)
  chunks = text_splitter.split_documents(documents)
  # Index each chunk to maintain unique identifiers
  for idx, chunk in enumerate(chunks):
      chunk.metadata['id'] = str(idx)
  return chunks

chunks = split_documents(documents=documents)

print(len(documents))
print(len(chunks))

In [7]:
def initialize_vectordb(persist_directory, embedding_function, collection_name):
    """Initialize the Chroma vector database, either loading an existing one or creating a new one."""
    if os.path.exists(persist_directory):
        return Chroma(persist_directory=persist_directory,
                      embedding_function=embedding_function,
                      collection_name=collection_name)
    else:
        return Chroma(embedding_function=embedding_function,
                      persist_directory=persist_directory,
                      collection_name=collection_name)

# Initialize vector database, using OpenAI embeddings
embedding = OpenAIEmbeddings(api_key=setup.get_openai_key(), model=MODEL)
vectordb = initialize_vectordb(PATH_DB_GCAI, embedding, COLLECTION_NAME)

# If you get an error, run this cell again.(TODO fix it)

In [ ]:
# Modified from https://github.com/pixegami/rag-tutorial-v2/blob/main/populate_database.py
def add_new_documents(vectordb, documents, batch_size=1000):
    """Add new documents to the database"""

    def load_existing_ids(vectordb):
        """Fetch existing document IDs from the database to avoid duplicates."""
        try:
            existing_items = vectordb.get(include=[])
            existing_ids = set(existing_items["ids"])
        except:
            existing_ids = set()
        return existing_ids

    existing_ids = load_existing_ids(vectordb)
    print(f"Number of existing documents in DB: {len(existing_ids)}")

    # Only add documents that don't exist in the DB.
    new_documents = []
    for document in documents:
        if document.metadata["id"] not in existing_ids:
            new_documents.append(document)

    if len(new_documents):
        print(f"Total new documents to add: {len(new_documents)}")

        # Process documents in batches
        for i in range(0, len(new_documents), batch_size):
            batch = new_documents[i:i + batch_size]
            batch_ids = [document.metadata["id"] for document in batch]
            vectordb.add_documents(batch, ids=batch_ids)
            print(f"Added batch {i//batch_size + 1} with {len(batch)} documents")
    else:
        print("No new documents to add")

add_new_documents(vectordb, chunks, batch_size=1000)

### Query the database

In [ ]:
## To read the db from file

# vectordb = Chroma(persist_directory=FN_DB_GCAI,
#                   embedding_function=OpenAIEmbeddings(api_key=OPENAI_API_KEY, model=MODEL),
#                   collection_name = COLLECTION_NAME
#                   )

In [ ]:
# Delete all items in the db

# items = vectordb.get(include=[])
# existing_ids = items["ids"]
# vectordb.delete(ids=existing_ids)

In [ ]:
# Retrieve metadata and document IDs from the database
items = vectordb.get(include=['metadatas'])
existing_ids = set(items["ids"])
metadata = items['metadatas']
print(f"Number of existing documents in DB: {len(existing_ids)}")
print(metadata[0])

In [11]:
# Set up a retriever for document querying
retriever = vectordb.as_retriever(search_kwargs={"k": 4})

In [ ]:
# Query the vector database using similarity search
query = 'Reuma'
docs = retriever.invoke(query)

print(f'Number of docs: {len(docs)}\n')
print(f'Retriever search type: {retriever.search_type}\n')

print(f'Documents most similar to "{query}":')
for doc in docs:
  print(doc.page_content)

In [15]:
# Initialize the QA chain for answering questions using the retrieved documents
qa_chain = RetrievalQA.from_chain_type(llm=OpenAI(api_key=setup.get_openai_key()),
                                  chain_type="stuff",
                                  retriever=retriever,
                                  return_source_documents=True)

In [36]:
# Function to test the pipeline with filtering based on ct_id
def query_retrieval_pipeline(query, ct_id=None):
    """Run a query through the retrieval pipeline with an optional filter on ct_id."""
    
    # Set up a filter for ct_id if provided
    if ct_id is not None:
        filter = {"ct_id": '1'}
        # Use the filter when setting up the retriever
        retriever = vectordb.as_retriever(search_kwargs={"k": 5}, filter=filter)
    else:
        retriever = vectordb.as_retriever(search_kwargs={"k": 4})
    
    # Set up the QA chain with the filtered retriever
    qa_chain_with_filter = RetrievalQA.from_chain_type(
        llm=OpenAI(api_key=setup.get_openai_key()),
        chain_type="stuff",
        retriever=retriever,
        return_source_documents=True
    )
    
    # Run the query through the QA chain
    llm_response = qa_chain_with_filter.invoke(query)
    
    # Process the LLM response
    result = llm_response.get('result', 'No result found')
    source_documents = llm_response.get('source_documents', [])
    
    # Display the result
    print(100 * '*')
    print(f"\nQuery: {query}")
    if ct_id is not None:
        print(f"Filter: Only results for ct_id = {ct_id}")
    print(f"\nResult:\n{result}")
    
    # Display the sources used for the response
    if source_documents:
        print("\nSources:")
        for i, source in enumerate(source_documents, 1):
            name = source.metadata.get('name', 'Unknown')
            datetime = source.metadata.get('datetime', 'Unknown')
            ct_id = source.metadata.get('ct_id', 'Unknown')
            print(f"{i}. Source: {name}, Date: {datetime}, Client ID: {ct_id}")
    else:
        print("\nNo source documents found.")
    
    print(100 * '*')

In [ ]:
# Example usage, filtering by ct_id = 1
query_retrieval_pipeline("Welke hulp krijgt client bij de ALD?", ct_id=1)